# 🚀 LLM Quantization Profiling & Benchmark Evaluation (Google Colab Acceleration)

This notebook provides a turnkey environment on **Google Colab (T4 / A100 / L4 GPU)** for **Standard Strided PPL Evaluation** and **Pre-Quantization Activation Outlier Profiling**.

### 💡 Key Benefits
1. **Dedicated Accelerator Memory**: Dedicated GPU memory prevents host memory contention.
2. **Full 2048-Token Context Length**: Standard T4 (16GB VRAM) comfortably supports 2048-token context windows for 1B-4B models.
3. **Interactive Visualization**: In-notebook plotting of layer-wise activation outlier spikes and peak-to-average ratios (PAR).

---


## 1. Verify GPU Hardware (NVIDIA-SMI)


In [ ]:
!nvidia-smi

## 2. Mount Google Drive (Optional Persistent Storage)


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!mkdir -p /content/drive/MyDrive/model_quantization_results


## 3. Environment & Dependency Setup


In [ ]:
# If cloning from GitHub, replace with repository URL:
# !git clone https://github.com/your-username/model_quantization.git
# %cd model_quantization

# Install core dependencies
!pip install -q "transformers>=4.40.0" "accelerate>=0.28.0" "datasets>=2.18.0" sentencepiece protobuf scipy matplotlib tqdm


## 4. Hugging Face Authentication (For Gated Models)
Authenticate using Hugging Face access token for gated models (LLaMA 3.2 / Gemma 2).


In [ ]:
from huggingface_hub import login
# login('hf_xxxxxxxxxxxxxxxxxxxx')


## 5. Text Generation & Inference Benchmark
Verifies text inference speed and generation latency under CUDA acceleration.


In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

model_id = "Qwen/Qwen3.5-2B"
device = "cuda" if torch.cuda.is_available() else "cpu"

print(f"Loading {model_id} to {device}...")
tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype=torch.bfloat16,
    device_map="auto",
    trust_remote_code=True
)

prompt = "Summarize three major advantages of Post-Training Quantization (PTQ) in LLM deployment:"
inputs = tokenizer(prompt, return_tensors="pt").to(device)

with torch.no_grad():
    outputs = model.generate(**inputs, max_new_tokens=150, temperature=0.7, do_sample=True)

print("
--- Generation Output ---")
print(tokenizer.decode(outputs[0], skip_special_tokens=True))


## 6. Official Standard WikiText-2 PPL (Perplexity) Benchmark
Strictly follows Hugging Face official Strided Sliding Window algorithm (2048 Context Length, 512 Stride, -100 Context Masking).


In [ ]:
import math
import time
import torch
from datasets import load_dataset
from tqdm import tqdm

def run_colab_ppl_benchmark(
    model,
    tokenizer,
    device="cuda",
    max_length=2048,
    stride=512,
    max_steps=None,
    dataset_name="wikitext",
    dataset_config="wikitext-2-raw-v1",
    split="test"
):
    dataset = load_dataset(dataset_name, dataset_config, split=split)
    full_text = "

".join([t for t in dataset["text"] if t.strip()])
    encodings = tokenizer(full_text, return_tensors="pt")
    seq_len = encodings.input_ids.size(1)

    print(f"Corpus Tokens: {seq_len:,}")
    print(f"Context Window: {max_length}, Sliding Stride: {stride}")

    nlls = []
    prev_end_loc = 0
    step_count = 0
    start_time = time.time()

    total_possible = (seq_len + stride - 1) // stride
    total_steps = min(total_possible, max_steps) if max_steps else total_possible
    pbar = tqdm(total=total_steps, desc="WikiText-2 PPL Benchmark")

    for begin_loc in range(0, seq_len, stride):
        end_loc = min(begin_loc + max_length, seq_len)
        trg_len = end_loc - prev_end_loc

        input_ids = encodings.input_ids[:, begin_loc:end_loc].to(device)
        target_ids = input_ids.clone()
        target_ids[:, :-trg_len] = -100

        with torch.no_grad():
            outputs = model(input_ids=input_ids, labels=target_ids)
            neg_log_likelihood = outputs.loss.item() * trg_len
            nlls.append(neg_log_likelihood)

        prev_end_loc = end_loc
        step_count += 1
        pbar.update(1)

        del outputs, input_ids, target_ids
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

        if end_loc == seq_len or (max_steps and step_count >= max_steps):
            break

    pbar.close()
    total_evaluated_tokens = prev_end_loc
    total_nll = sum(nlls)
    ppl = math.exp(total_nll / total_evaluated_tokens) if total_evaluated_tokens > 0 else float('nan')
    elapsed = time.time() - start_time

    print("
" + "=" * 60)
    print(f"🏆 WikiText-2 PPL (Perplexity) : {ppl:.4f}")
    print(f"⚡ Total Elapsed Time          : {elapsed:.2f} s")
    print(f"🚀 Processing Throughput       : {total_evaluated_tokens / elapsed:.2f} tokens/s")
    print(f"📝 Evaluated Corpus Tokens     : {total_evaluated_tokens:,}")
    print("=" * 60)
    return ppl, elapsed

ppl_score, elapsed_time = run_colab_ppl_benchmark(
    model=model,
    tokenizer=tokenizer,
    device=device,
    max_length=2048,
    stride=512,
    max_steps=40
)


## 7. Activation Outlier Profiling & Visualization
Captures layer-wise activation tensors via forward hooks to measure channel spikes and peak-to-average ratios (PAR).


In [ ]:
import matplotlib.pyplot as plt

def find_decoder_layers(model):
    for name, module in model.named_modules():
        if isinstance(module, torch.nn.ModuleList) and len(module) > 5:
            if "language_model" in name or ("layers" in name and "vision" not in name and "audio" not in name):
                return list(module)
    raise RuntimeError("Unable to locate Transformer Decoder Layers")

layers = find_decoder_layers(model)
num_layers = len(layers)
records = {}
hooks = []

def make_hook(idx):
    def hook_fn(module, args, output):
        feat = output[0] if isinstance(output, tuple) else output
        feat = feat.detach().float().abs()
        g_max = feat.max().item()
        g_mean = feat.mean().item()
        par = g_max / (g_mean + 1e-8)
        records[idx] = {"layer": idx, "max": g_max, "mean": g_mean, "par": par}
    return hook_fn

for i, layer in enumerate(layers):
    hooks.append(layer.register_forward_hook(make_hook(i)))

# Single forward calibration pass
sample_input = tokenizer("Large language models exhibit significant activation outliers in deeper layers.", return_tensors="pt").to(device)
with torch.no_grad():
    model(**sample_input)

for h in hooks:
    h.remove()

layer_indices = [records[i]["layer"] for i in range(num_layers)]
pars = [records[i]["par"] for i in range(num_layers)]
max_vals = [records[i]["max"] for i in range(num_layers)]

plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1)
plt.plot(layer_indices, pars, marker='o', color='#e74c3c', linewidth=2)
plt.axhline(y=15, color='gray', linestyle='--', label='Outlier Threshold (15x)')
plt.title(f"{model_id} Layer-wise Peak-to-Average Ratio (PAR)")
plt.xlabel("Layer Index")
plt.ylabel("PAR (Max / Mean)")
plt.grid(True, alpha=0.3)
plt.legend()

plt.subplot(1, 2, 2)
plt.bar(layer_indices, max_vals, color='#3498db', alpha=0.8)
plt.title(f"{model_id} Layer-wise Global Max Activation")
plt.xlabel("Layer Index")
plt.ylabel("Max Activation Value")
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()


## 8. Persist Benchmark Results (CSV Export)
Logs benchmark metrics to CSV on Google Drive for persistence across sessions.


In [ ]:
import pandas as pd

df_results = pd.DataFrame([{
    "model_id": model_id,
    "ppl_wikitext2": ppl_score,
    "elapsed_sec": elapsed_time,
    "max_par": max(pars),
    "max_activation_value": max(max_vals),
}])

csv_path = "/content/drive/MyDrive/model_quantization_results/benchmark_summary.csv"
df_results.to_csv(csv_path, mode="a", header=not pd.io.common.file_exists(csv_path), index=False)
print(f"✅ Benchmark record successfully saved to: {csv_path}")
display(df_results)
